In [ ]:
# =============================================================================
# CÉLULA 0: PARÂMETROS GLOBAIS (CONFIGURÁVEIS POR REGIME)
# =============================================================================
EMAIL_REMETENTE = "ab11.94958191@gmail.com"
SENHA_APP = ""

PARAMS_BAIXA_VOL = {
    'kelly_frac': 0.30,
    'wyckoff_threshold': 0.75,
    'gap_max_pct': 0.03,
    'custos_pct': 0.003,
    'exigir_volume_anormal': False
}

PARAMS_ALTA_VOL = {
    'kelly_frac': 0.15,
    'wyckoff_threshold': 0.85,
    'gap_max_pct': 0.015,
    'custos_pct': 0.006,
    'exigir_volume_anormal': True
}

# ✅ Inicializa como cópia (será atualizado via .clear()/.update() na Célula 4)
PARAMS_ATIVOS = PARAMS_BAIXA_VOL.copy()

MAX_SETUPS_POR_DIA = 5
MAX_PERDAS_CONSECUTIVAS = 3
DRAWDOWN_MAX_DIARIO = 0.02
MAX_DIAS_LOG = 30

HABILITAR_LOGGING = True
ARQUIVO_LOG = "trading_log_v42.json"
CAPITAL_TOTAL = 100000.0
WIN_RATE_ESTIMADO = 0.40
PAYOFF_ESTIMADO = 3.0

SETORES_BLOQUEADOS = ['AEREA']
TICKERS_BLOQUEADOS = ['GFSA3.SA', 'ONCO3.SA', 'PMAM3.SA', 'AZTE3.SA', 'RAIZ4.SA',
                      'BHIA3.SA', 'CASH3.SA', 'LJQQ3.SA', 'RCSL4.SA', 'HBOR3.SA']

FALLBACK_TICKERS = [
    'PETR4.SA', 'VALE3.SA', 'ITUB4.SA', 'BBDC4.SA', 'BBAS3.SA', 'ABEV3.SA',
    'WEGE3.SA', 'RADL3.SA', 'SUZB3.SA', 'GGBR4.SA', 'MGLU3.SA', 'VVAR3.SA',
    'RENT3.SA', 'RAIL3.SA', 'CCRO3.SA', 'ELET3.SA', 'CPFE3.SA', 'SBSP3.SA',
    'SANB11.SA', 'B3SA3.SA', 'JBSS3.SA', 'BRFS3.SA', 'KLBN11.SA', 'EQTL3.SA'
]


In [ ]:
# =============================================================================
# CÉLULA 1: INSTALAÇÃO, IMPORTAÇÕES E LOGGING COM ROTAÇÃO
# =============================================================================
!pip install yfinance pandas-ta optuna --quiet

import yfinance as yf
import pandas as pd
import numpy as np
import pandas_ta as ta
import requests
from bs4 import BeautifulSoup
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from datetime import datetime, timedelta
import time, warnings, json, os
from typing import Optional, Tuple, Dict, List

warnings.filterwarnings("ignore")

def log_evento(tipo: str, ticker: str, dados: dict, arquivo: str = ARQUIVO_LOG, max_dias: int = MAX_DIAS_LOG):
    if not HABILITAR_LOGGING:
        return
    registro = {'timestamp': datetime.now().isoformat(), 'tipo': tipo, 'ticker': ticker, 'dados': dados}
    logs = []
    if os.path.exists(arquivo):
        try:
            with open(arquivo, 'r', encoding='utf-8') as f:
                logs = json.load(f)
        except:
            logs = []
    # Rotacionar: remover registros antigos
    cutoff = datetime.now() - timedelta(days=max_dias)
    logs = [l for l in logs if datetime.fromisoformat(l['timestamp']) > cutoff]
    logs.append(registro)
    with open(arquivo, 'w', encoding='utf-8') as f:
        json.dump(logs, f, ensure_ascii=False, indent=2)

print("✅ Bibliotecas instaladas e logging com rotação configurado.")


In [ ]:
# =============================================================================
# CÉLULA 2: FUNÇÕES AUXILIARES (WYCKOFF CORRIGIDO + LOG-SAFE)
# =============================================================================

def calcular_eficiencia_candle(df: pd.DataFrame) -> pd.Series:
    corpo = abs(df['Close'] - df['Open'])
    sombra_sup = df['High'] - df[['Close', 'Open']].max(axis=1)
    sombra_inf = df[['Close', 'Open']].min(axis=1) - df['Low']
    range_total = df['High'] - df['Low'].replace(0, np.nan)
    eficiencia = pd.Series(index=df.index, dtype=float)
    alta, baixa = df['Close'] > df['Open'], df['Close'] < df['Open']
    eficiencia[alta] = 1 - (sombra_sup[alta] / range_total[alta])
    eficiencia[baixa] = 1 - (sombra_inf[baixa] / range_total[baixa])
    return eficiencia

def detectar_regime(df: pd.DataFrame, janela: int = 20) -> pd.Series:
    df_temp = df.copy()
    df_temp['retorno'] = df_temp['Close'].pct_change()
    df_temp['volatilidade'] = df_temp['retorno'].rolling(janela).std()
    adx = ta.adx(df_temp['High'], df_temp['Low'], df_temp['Close'], length=14)
    df_temp['adx'] = adx['ADX_14']
    df_temp = df_temp.dropna(subset=['volatilidade', 'adx'])
    if df_temp.empty:
        return pd.Series(index=df.index, dtype=int)
    v_p33, v_p67 = df_temp['volatilidade'].quantile([0.33, 0.67])
    a_p33, a_p67 = df_temp['adx'].quantile([0.33, 0.67])
    def classificar(row):
        v, a = row['volatilidade'], row['adx']
        if v < v_p33 and a < a_p33: return 0
        elif v > v_p67 or a > a_p67: return 2
        else: return 1
    regimes = df_temp.apply(classificar, axis=1)
    regime_series = pd.Series(index=df.index, dtype=int)
    regime_series.loc[regimes.index] = regimes
    regime_series.ffill(inplace=True)
    return regime_series

# PIVÔS ANTI-REPAINTING (CONFIRMAÇÃO DE 1 CANDLE)
def detectar_swing_low(df: pd.DataFrame, janela: int = 10, confirmar: bool = True) -> Optional[float]:
    lows, closes = df['Low'].values, df['Close'].values
    swing_lows = []
    for i in range(janela, len(lows)):
        if i - janela >= 0 and lows[i] <= min(lows[i-janela:i]):
            if not confirmar or (i+1 < len(lows) and closes[i+1] > lows[i]):
                swing_lows.append(lows[i])
    return float(swing_lows[-1]) if swing_lows else float(df['Low'].min())

def detectar_swing_high(df: pd.DataFrame, janela: int = 10, confirmar: bool = True) -> Optional[float]:
    highs, closes = df['High'].values, df['Close'].values
    swing_highs = []
    for i in range(janela, len(highs)):
        if i - janela >= 0 and highs[i] >= max(highs[i-janela:i]):
            if not confirmar or (i+1 < len(highs) and closes[i+1] < highs[i]):
                swing_highs.append(highs[i])
    return float(swing_highs[-1]) if swing_highs else float(df['High'].max())

# ✅ LOG-SAFE: Proteção contra np.log(0)
def calcular_lta_pivos(df: pd.DataFrame, janela_pivo: int = 5) -> Optional[float]:
    lows = df['Low'].values
    lows_seguro = np.where(lows <= 0, np.nan, lows)
    log_lows = np.log(lows_seguro)
    fundos = []
    for i in range(janela_pivo, len(log_lows) - janela_pivo):
        if np.isnan(log_lows[i]): continue
        if log_lows[i] == min(log_lows[i-janela_pivo:i+janela_pivo+1]):
            fundos.append((i, log_lows[i]))
    if len(fundos) >= 2:
        (x1, y1), (x2, y2) = fundos[-2], fundos[-1]
        x_atual = len(log_lows) - 1
        incl = (y2 - y1) / (x2 - x1)
        return np.exp(y2 + incl * (x_atual - x2))
    return None

def calcular_ltb_pivos(df: pd.DataFrame, janela_pivo: int = 5) -> Optional[float]:
    highs = df['High'].values
    highs_seguro = np.where(highs <= 0, np.nan, highs)
    log_highs = np.log(highs_seguro)
    topos = []
    for i in range(janela_pivo, len(log_highs) - janela_pivo):
        if np.isnan(log_highs[i]): continue
        if log_highs[i] == max(log_highs[i-janela_pivo:i+janela_pivo+1]):
            topos.append((i, log_highs[i]))
    if len(topos) >= 2 and topos[-2][1] > topos[-1][1]:
        (x1, y1), (x2, y2) = topos[-2], topos[-1]
        x_atual = len(log_highs) - 1
        incl = (y2 - y1) / (x2 - x1)
        return np.exp(y2 + incl * (x_atual - x2))
    return None

def detectar_padrao_altista(df_diario: pd.DataFrame) -> bool:
    if len(df_diario) < 3: return False
    u, p = df_diario.iloc[-1], df_diario.iloc[-2]
    c_u = abs(u['Close'] - u['Open'])
    r_u = u['High'] - u['Low']
    si_u = min(u['Close'], u['Open']) - u['Low']
    ss_u = u['High'] - max(u['Close'], u['Open'])
    if r_u > 0 and si_u >= 2 * c_u and ss_u <= 0.3 * c_u: return True
    if p['Close'] < p['Open'] and u['Close'] > u['Open']:
        if u['Open'] <= p['Close'] and u['Close'] >= p['Open']: return True
        meio = (p['Open'] + p['Close']) / 2
        if u['Open'] <= p['Close'] and u['Close'] >= meio: return True
    return False

def detectar_padrao_baixista(df_diario: pd.DataFrame) -> bool:
    if len(df_diario) < 3: return False
    u, p = df_diario.iloc[-1], df_diario.iloc[-2]
    c_u = abs(u['Close'] - u['Open'])
    r_u = u['High'] - u['Low']
    si_u = min(u['Close'], u['Open']) - u['Low']
    ss_u = u['High'] - max(u['Close'], u['Open'])
    if r_u > 0 and ss_u >= 2 * c_u and si_u <= 0.3 * c_u: return True
    if p['Close'] > p['Open'] and u['Close'] < u['Open']:
        if u['Open'] >= p['Close'] and u['Close'] <= p['Open']: return True
        meio = (p['Open'] + p['Close']) / 2
        if u['Open'] >= p['Close'] and u['Close'] <= meio: return True
    return False

# ✅ Sentimento VADER removido para limpeza de código e redução de falsos sinais
SENTIMENTO_PADRAO = 0.0

def detectar_volume_anormal(df: pd.DataFrame, periodo: int = 20, limiar: float = 1.5) -> bool:
    if len(df) < periodo: return False
    vol_medio = df['Volume'].rolling(periodo).mean().iloc[-1]
    return df['Volume'].iloc[-1] >= vol_medio * limiar if pd.notna(vol_medio) else False

def fractional_kelly(win_rate: float, payoff_ratio: float, frac: float = 0.25) -> float:
    if payoff_ratio <= 0: return 0.0
    kelly = (payoff_ratio * win_rate - (1 - win_rate)) / payoff_ratio
    return max(0.0, min(kelly, 0.25)) * frac

MACRO_REFERENCE = {
    'VALE3.SA': ('GC=F', 0.6), 'PETR4.SA': ('CL=F', 0.8), 'PETR3.SA': ('CL=F', 0.8),
    'CSNA3.SA': ('GC=F', 0.5), 'GGBR4.SA': ('GC=F', 0.5), 'CAML3.SA': ('WEAT', 0.4),
    'JBSS3.SA': ('WEAT', 0.5), 'ABEV3.SA': ('CORN', 0.3), 'RADL3.SA': ('XLP', 0.3),
    'PRIO3.SA': ('CL=F', 0.7),
}

def verificar_alinhamento_macro(ticker: str, direcao: str, cache_macro: dict) -> Tuple[bool, int]:
    if ticker not in MACRO_REFERENCE: return True, 15
    ref, _ = MACRO_REFERENCE[ticker]
    if ref not in cache_macro or cache_macro[ref] is None or len(cache_macro[ref]) < 55: return True, 15
    precos = cache_macro[ref]
    ema50 = pd.Series(precos).ewm(span=50, adjust=False).mean()
    e_at, e_lag = ema50.iloc[-1], ema50.iloc[-5]
    if pd.isna(e_at) or pd.isna(e_lag): return True, 15
    slope_pos = e_at > e_lag
    if direcao == 'COMPRA' and precos[-1] > e_at and slope_pos: return True, 30
    if direcao == 'VENDA' and precos[-1] < e_at and not slope_pos: return True, 30
    return False, 0

def avaliar_qualidade_volume(df_w: pd.DataFrame, direcao: str) -> Tuple[str, int]:
    vol_u = df_w['Volume'].iloc[-1]
    vol_m = df_w['Volume'].rolling(20).mean().iloc[-1]
    ef = df_w['Eficiencia'].iloc[-1] if 'Eficiencia' in df_w.columns else 0.5
    if pd.isna(vol_m) or vol_m == 0: return 'NEUTRO', 10
    if vol_u > vol_m * 1.5:
        if direcao == 'COMPRA' and ef > 0.7: return 'ALTA_CONVICCAO', 25
        if direcao == 'VENDA' and ef > 0.7: return 'ALTA_CONVICCAO', 25
        if ef < 0.5: return 'POSSIVEL_ARMADILHA', 15
    return 'NEUTRO', 10

# ✅ WYCKOFF ADAPTATIVO CORRIGIDO
def detectar_fase_wyckoff_adaptativo(df_w: pd.DataFrame, suporte: float = None, 
                                     resistencia: float = None, atr_period: int = 14) -> Tuple[str, float]:
    if len(df_w) < 30: return 'INDEFINIDO', 0.0
    atr = ta.atr(df_w['High'], df_w['Low'], df_w['Close'], length=atr_period)
    atr_med = atr.rolling(20).mean()
    vol_rel = atr.iloc[-1] / atr_med.iloc[-1] if pd.notna(atr_med.iloc[-1]) else 1.0
    thr_base = PARAMS_ATIVOS.get('wyckoff_threshold', 0.75)
    thr_comp = max(0.65, min(0.90, thr_base + 0.10 * (vol_rel - 1)))
    
    range_semanal = df_w['High'] - df_w['Low']
    r_med = range_semanal.rolling(20).mean().iloc[-1]
    if range_semanal.iloc[-1] >= r_med * thr_comp: return 'INDEFINIDO', 0.0
    
    px = df_w['Close'].iloc[-1]
    if suporte is None: suporte = df_w['Low'].rolling(20).min().iloc[-1]
    if resistencia is None: resistencia = df_w['High'].rolling(20).max().iloc[-1]
    faixa = resistencia - suporte
    if faixa == 0: return 'INDEFINIDO', 0.0
    
    pos_rel = (px - suporte) / faixa
    candles = df_w.iloc[-8:]
    alta, baixa = candles['Close'] > candles['Open'], candles['Close'] < candles['Open']
    vol_alta = candles.loc[alta, 'Volume'].mean() if alta.any() else 0
    vol_baixa = candles.loc[baixa, 'Volume'].mean() if baixa.any() else 0
    if vol_baixa == 0: return 'INDEFINIDO', 0.0
    
    razao = vol_alta / vol_baixa
    conf = min(1.0, (r_med - range_semanal.iloc[-1]) / r_med) if r_med > 0 else 0.5
    if pos_rel < 0.4 and razao > 1.2: return 'ACUMULACAO', conf
    if pos_rel > 0.6 and razao < 0.8: return 'DISTRIBUICAO', conf
    return 'INDEFINIDO', 0.0

def calcular_alvos_fibonacci(df: pd.DataFrame, direcao: str, fib_window: int = 20) -> Dict[str, float]:
    if len(df) < fib_window: return {}
    try:
        df_rec = df.iloc[-fib_window:]
        sl = detectar_swing_low(df_rec, janela=5, confirmar=False)
        sh = detectar_swing_high(df_rec, janela=5, confirmar=False)
        if sl is None or sh is None or sl >= sh: return {}
        amp = np.log(sh) - np.log(sl)
        base = np.log(sh)
        mults = [1.000, 1.618, 2.618, 4.236]
        if direcao == 'COMPRA':
            return {f'{m*100}%': round(np.exp(base + amp * m), 2) for m in mults}
        return {f'{m*100}%': round(np.exp(base - amp * m), 2) for m in mults}
    except: return {}

def calcular_score_confianca(setup: dict, alinhado_macro: bool, qualidade_volume: str, 
                             score_volume: int, fase_wyckoff: str, wyckoff_conf: float = 1.0) -> float:
    score = (30 if alinhado_macro else 0) + score_volume
    if setup['Eficiência'] and setup['Eficiência'] > 0.8: score += 20
    elif setup['Eficiência'] and setup['Eficiência'] > 0.6: score += 10
    if setup['Regime'] == 2: score += 10
    elif setup['Regime'] == 1: score += 5
    if setup['Direcao'] == 'COMPRA' and fase_wyckoff == 'ACUMULACAO': score += int(20 * wyckoff_conf)
    elif setup['Direcao'] == 'VENDA' and fase_wyckoff == 'DISTRIBUICAO': score += int(20 * wyckoff_conf)
    return min(score, 100) / 100.0

def calcular_alvo_recomendado(setup: dict, margem: float = 0.03) -> Tuple[float, str]:
    fibos = setup.get('Alvos Fibonacci', {})
    if not fibos or '161.8%' not in fibos: return setup['Alvo 3:1'], '3:1'
    fib = fibos['161.8%']
    if setup['Direcao'] == 'COMPRA' and fib <= setup['Resistência'] * (1 + margem): return fib, 'Fib 161.8%'
    if setup['Direcao'] == 'VENDA' and fib >= setup['Suporte'] * (1 - margem): return fib, 'Fib 161.8%'
    return setup['Alvo 3:1'], '3:1'

def calcular_payoff_real(entrada: float, alvo: float, stop: float, custos_pct: float) -> float:
    risco = abs(entrada - stop)
    if risco == 0: return 0.0
    retorno = abs(alvo - entrada) - (entrada * custos_pct * 2)
    return round(max(0, retorno) / risco, 2)

# ✅ CORREÇÃO: Aceita DataFrame ou Series com coluna 'Close'
def detectar_regime_volatilidade(df: pd.DataFrame, janela: int = 40) -> str:
    if isinstance(df, pd.Series): df = pd.DataFrame({'Close': df})
    if len(df) < janela or 'Close' not in df.columns: return 'BAIXA'
    ret = df['Close'].pct_change().dropna()
    vol_at = ret.rolling(20).std().iloc[-1]
    vol_hist = ret.rolling(janela).std().dropna()
    if pd.isna(vol_at) or vol_hist.empty: return 'BAIXA'
    return 'ALTA' if (vol_hist < vol_at).mean() > 0.7 else 'BAIXA'

print("✅ Célula 2 carregada (Wyckoff corrigido, log-safe, sem VADER).")


In [ ]:
# =============================================================================
# CÉLULA 3: FUNÇÕES DE ANÁLISE (SENTIMENTO DESATIVADO)
# =============================================================================

OTIMIZADO_SWING = {'atr_period': 14, 'atr_mult': 1.8, 'swing_window': 12, 'lta_pivo_window': 6, 'ltb_pivo_window': 6, 'mm200_semanal': True, 'mm200_diaria': True}
OTIMIZADO_POSITION = {'atr_period': 14, 'atr_mult': 2.5, 'swing_window': 24, 'lta_pivo_window': 12, 'ltb_pivo_window': 12, 'mm50_mensal': True}

def analisar_swing_trade(ticker: str, df_w: pd.DataFrame = None, df_d: pd.DataFrame = None) -> Optional[List[dict]]:
    try:
        if df_w is None: return None
        df_w = df_w.copy()
        if not set(['Close','High','Low','Open','Volume']).issubset(df_w.columns):
            df_w.rename(columns={k:v for k,v in {'close':'Close','high':'High','low':'Low','open':'Open','volume':'Volume'}.items() if k in df_w.columns}, inplace=True)
        df_w.sort_index(inplace=True)
        if not isinstance(df_w.index, pd.DatetimeIndex): df_w.index = pd.to_datetime(df_w.index)
        
        df_w['Eficiencia'] = calcular_eficiencia_candle(df_w)
        df_w['Regime'] = detectar_regime(df_w)
        ult = df_w.iloc[-1]
        entrada = float(ult['Close'])
        if pd.isna(entrada) or entrada <= 0: return None
        
        rh = float(df_w['High'].rolling(window=min(52, len(df_w))).max().iloc[-1])
        rl = float(df_w['Low'].rolling(window=min(52, len(df_w))).min().iloc[-1])
        reg = int(ult['Regime']) if not pd.isna(ult['Regime']) else -1
        ef = round(float(ult['Eficiencia']), 2) if not pd.isna(ult['Eficiencia']) else None
        atr = float(ta.atr(df_w['High'], df_w['Low'], df_w['Close'], length=OTIMIZADO_SWING['atr_period']).iloc[-1] or 0.0)
        
        pa = pb = True
        if df_d is not None and not df_d.empty:
            df_loc = df_d.copy()
            if isinstance(df_loc.columns, pd.MultiIndex): df_loc.columns = df_loc.columns.droplevel(1)
            df_loc.columns = ['Close', 'High', 'Low', 'Open', 'Volume']
            pa = detectar_padrao_altista(df_loc)
            pb = detectar_padrao_baixista(df_loc)
            
        sentimento = SENTIMENTO_PADRAO
        vol_an = detectar_volume_anormal(df_w, periodo=20, limiar=1.5)
        setups = []
        
        if pa:
            fib = calcular_alvos_fibonacci(df_w, 'COMPRA')
            st_atr = entrada - (OTIMIZADO_SWING['atr_mult'] * atr) if atr > 0 else None
            st_sw = detectar_swing_low(df_w, janela=OTIMIZADO_SWING['swing_window'])
            st_sw = st_sw if st_sw < entrada else None
            st_lta = calcular_lta_pivos(df_w, janela_pivo=OTIMIZADO_SWING['lta_pivo_window'])
            st_lta = st_lta if st_lta and st_lta < entrada else None
            for met, sl in [('ATR', st_atr), ('Swing Low', st_sw), ('LTA Pivôs', st_lta)]:
                if sl is None or sl <= 0 or sl >= entrada: continue
                r = entrada - sl
                al = entrada + (r * 3)
                if al <= rh * 1.05:
                    setups.append({'Ticker': ticker, 'Modalidade': 'Swing', 'Direcao': 'COMPRA', 'Entrada': round(entrada,2), 'Método Stop': met, 'Stop Loss': round(sl,2), 'Risco (R$)': round(r,2), 'Alvo 3:1': round(al,2), 'Resistência': round(rh,2), 'Suporte': round(rl,2), 'Regime': reg, 'Eficiência': ef, 'Sentimento': sentimento, 'Volume Anormal': vol_an, 'Alvos Fibonacci': fib})
                    
        if pb:
            fib = calcular_alvos_fibonacci(df_w, 'VENDA')
            st_atr = entrada + (OTIMIZADO_SWING['atr_mult'] * atr) if atr > 0 else None
            st_sw = detectar_swing_high(df_w, janela=OTIMIZADO_SWING['swing_window'])
            st_sw = st_sw if st_sw > entrada else None
            st_ltb = calcular_ltb_pivos(df_w, janela_pivo=OTIMIZADO_SWING['ltb_pivo_window'])
            st_ltb = st_ltb if st_ltb and st_ltb > entrada else None
            for met, sl in [('ATR', st_atr), ('Swing High', st_sw), ('LTB Pivôs', st_ltb)]:
                if sl is None or sl <= entrada: continue
                r = sl - entrada
                al = entrada - (r * 3)
                if al >= rl * 0.95:
                    setups.append({'Ticker': ticker, 'Modalidade': 'Swing', 'Direcao': 'VENDA', 'Entrada': round(entrada,2), 'Método Stop': met, 'Stop Loss': round(sl,2), 'Risco (R$)': round(r,2), 'Alvo 3:1': round(al,2), 'Resistência': round(rh,2), 'Suporte': round(rl,2), 'Regime': reg, 'Eficiência': ef, 'Sentimento': sentimento, 'Volume Anormal': vol_an, 'Alvos Fibonacci': fib})
        return setups if setups else None
    except Exception as e:
        log_evento('ERRO', ticker, {'funcao': 'analisar_swing_trade', 'erro': str(e)})
        return None

def analisar_position_trade(ticker: str, df_m: pd.DataFrame = None, df_w: pd.DataFrame = None) -> Optional[List[dict]]:
    try:
        if df_m is None: return None
        df_m = df_m.copy()
        if not set(['Close','High','Low','Open','Volume']).issubset(df_m.columns):
            df_m.rename(columns={k:v for k,v in {'close':'Close','high':'High','low':'Low','open':'Open','volume':'Volume'}.items() if k in df_m.columns}, inplace=True)
        df_m.sort_index(inplace=True)
        if not isinstance(df_m.index, pd.DatetimeIndex): df_m.index = pd.to_datetime(df_m.index)
        
        df_m['Eficiencia'] = calcular_eficiencia_candle(df_m)
        df_m['Regime'] = detectar_regime(df_m)
        ult = df_m.iloc[-1]
        entrada = float(ult['Close'])
        if pd.isna(entrada) or entrada <= 0: return None
        
        lb = min(60, len(df_m))
        rh = float(df_m['High'].rolling(window=lb).max().iloc[-1])
        rl = float(df_m['Low'].rolling(window=lb).min().iloc[-1])
        reg = int(ult['Regime']) if not pd.isna(ult['Regime']) else -1
        ef = round(float(ult['Eficiencia']), 2) if not pd.isna(ult['Eficiencia']) else None
        atr = float(ta.atr(df_m['High'], df_m['Low'], df_m['Close'], length=OTIMIZADO_POSITION['atr_period']).iloc[-1] or 0.0)
        
        sentimento = SENTIMENTO_PADRAO
        vol_an = detectar_volume_anormal(df_m, periodo=20, limiar=1.5)
        setups = []
        
        fib_c = calcular_alvos_fibonacci(df_w, 'COMPRA') if df_w is not None else {}
        st_atr = entrada - (OTIMIZADO_POSITION['atr_mult'] * atr) if atr > 0 else None
        st_sw = detectar_swing_low(df_m, janela=OTIMIZADO_POSITION['swing_window'])
        st_sw = st_sw if st_sw and st_sw < entrada else None
        st_lta = calcular_lta_pivos(df_m, janela_pivo=OTIMIZADO_POSITION['lta_pivo_window'])
        st_lta = st_lta if st_lta and st_lta < entrada else None
        for met, sl in [('ATR', st_atr), ('Swing Low', st_sw), ('LTA Pivôs', st_lta)]:
            if sl is None or sl <= 0 or sl >= entrada: continue
            r = entrada - sl
            al = entrada + (r * 3)
            if al <= rh * 1.10:
                setups.append({'Ticker': ticker, 'Modalidade': 'Position', 'Direcao': 'COMPRA', 'Entrada': round(entrada,2), 'Método Stop': met, 'Stop Loss': round(sl,2), 'Risco (R$)': round(r,2), 'Alvo 3:1': round(al,2), 'Resistência': round(rh,2), 'Suporte': round(rl,2), 'Regime': reg, 'Eficiência': ef, 'Sentimento': sentimento, 'Volume Anormal': vol_an, 'Alvos Fibonacci': fib_c})
                
        fib_v = calcular_alvos_fibonacci(df_w, 'VENDA') if df_w is not None else {}
        st_atr = entrada + (OTIMIZADO_POSITION['atr_mult'] * atr) if atr > 0 else None
        st_sw = detectar_swing_high(df_m, janela=OTIMIZADO_POSITION['swing_window'])
        st_sw = st_sw if st_sw and st_sw > entrada else None
        st_ltb = calcular_ltb_pivos(df_m, janela_pivo=OTIMIZADO_POSITION['ltb_pivo_window'])
        st_ltb = st_ltb if st_ltb and st_ltb > entrada else None
        for met, sl in [('ATR', st_atr), ('Swing High', st_sw), ('LTB Pivôs', st_ltb)]:
            if sl is None or sl <= entrada: continue
            r = sl - entrada
            al = entrada - (r * 3)
            if al >= rl * 0.90:
                setups.append({'Ticker': ticker, 'Modalidade': 'Position', 'Direcao': 'VENDA', 'Entrada': round(entrada,2), 'Método Stop': met, 'Stop Loss': round(sl,2), 'Risco (R$)': round(r,2), 'Alvo 3:1': round(al,2), 'Resistência': round(rh,2), 'Suporte': round(rl,2), 'Regime': reg, 'Eficiência': ef, 'Sentimento': sentimento, 'Volume Anormal': vol_an, 'Alvos Fibonacci': fib_v})
        return setups if setups else None
    except Exception as e:
        log_evento('ERRO', ticker, {'funcao': 'analisar_position_trade', 'erro': str(e)})
        return None

print("✅ Célula 3 carregada (Sentimento desativado, análise bidirecional).")


In [ ]:
# =============================================================================
# CÉLULA 4: EXECUÇÃO PRINCIPAL (CORRIGIDA: ESCOPO, DOWNLOAD 5Y, RESAMPLE W-FRI)
# =============================================================================

try:
    from google.colab import userdata
    if not SENHA_APP: SENHA_APP = userdata.get('GMAIL_APP_PASSWORD')
except: pass

VOLUME_MINIMO_ACAO = 1_000_000
VOLUME_FINANCEIRO_MINIMO = 1_000_000
LIMITE_LIQUIDEZ_FINANCEIRA = 5_000_000
PRECO_MINIMO = 5.00
RISCO_PERCENTUAL_MINIMO = 0.02
RISCO_PERCENTUAL_MAXIMO = 0.20
EXIGIR_CONFLUENCIA = True

def montar_tabela_html(oportunidades: List[dict], titulo: str, regime_vol: str) -> str:
    if not oportunidades: return ""
    corpo = f"<h3>{titulo}</h3><table border='1' cellpadding='4' cellspacing='0' style='border-collapse:collapse;'>"
    corpo += "<tr><th>Ticker</th><th>Dir.</th><th>Entrada</th><th>Stop</th><th>Alvo Rec.</th><th>Método</th><th>Payoff Real</th><th>Vol. Anormal</th><th>Lote</th><th>Score</th></tr>"
    for op in oportunidades:
        vol_an_icon = '✅' if op.get('Volume Anormal') else '❌'
        corpo += f"<tr><td>{op['Ticker']}</td><td>{op['Direcao']}</td><td>R$ {op['Entrada']:.2f}</td><td>R$ {op['Stop Loss']:.2f}</td><td>R$ {op['Alvo Recomendado']:.2f}</td><td>{op['Método Alvo']}</td><td>{op['Payoff Real']}:1</td><td>{vol_an_icon}</td><td>{op['Lote']}</td><td>{op.get('Score', 'N/A')}</td></tr>"
    corpo += f"</table><br><p><small>Custos: {PARAMS_ATIVOS['custos_pct']*100:.1f}% | Regime: {regime_vol}</small></p>"
    return corpo

def enviar_email_ou_exibir(oportunidades: List[dict], modalidade: str, regime_vol: str):
    if not oportunidades:
        print(f"ℹ️ Nenhuma oportunidade de {modalidade} encontrada.")
        return
    if EMAIL_REMETENTE and SENHA_APP:
        try:
            msg = MIMEMultipart()
            msg['From'] = EMAIL_REMETENTE
            msg['To'] = EMAIL_REMETENTE
            msg['Subject'] = f"🚨 Oportunidades {modalidade} - {datetime.now().strftime('%d/%m/%Y')}"
            msg.attach(MIMEText(montar_tabela_html(oportunidades, "Setups Aprovados", regime_vol), 'html'))
            with smtplib.SMTP_SSL('smtp.gmail.com', 465) as server:
                server.login(EMAIL_REMETENTE, SENHA_APP)
                server.send_message(msg)
            print(f"✅ E-mail ({modalidade}) enviado.")
        except Exception as e:
            print(f"❌ Falha no e-mail: {e}")
            log_evento('ERRO_EMAIL', 'SISTEMA', {'erro': str(e)})
    else:
        print(f"📧 E-mail não configurado. Exibindo na tela.")
    df_op = pd.DataFrame(oportunidades)
    cols = ['Ticker', 'Direcao', 'Entrada', 'Método Stop', 'Stop Loss', 'Risco (R$)', 'Alvo 3:1', 'Alvo Recomendado', 'Payoff Real', 'Resistência', 'Suporte', 'Regime', 'Eficiência', 'Volume Anormal', 'Fase Wyckoff', 'Score', 'Kelly %', 'Lote']
    try:
        from IPython.display import display
        display(df_op[cols].sort_values(['Direcao', 'Ticker']))
    except:
        print(df_op[cols].sort_values(['Direcao', 'Ticker']).to_string())
    csv_name = f"oportunidades_{modalidade.lower()}_{datetime.now().strftime('%Y%m%d')}.csv"
    df_op[cols].to_csv(csv_name, index=False)
    try:
        from google.colab import files
        files.download(csv_name)
    except: print(f"📁 Arquivo '{csv_name}' salvo.")

def obter_tickers_b3() -> List[str]:
    try:
        url = "https://www.dadosdemercado.com.br/acoes"
        soup = BeautifulSoup(requests.get(url, timeout=10).content, 'html.parser')
        tickers = [row.find_all('td')[0].text.strip() for row in soup.select('table tbody tr') if row.find_all('td') and not row.find_all('td')[0].text.strip().startswith('#')]
        return tickers if tickers else FALLBACK_TICKERS.copy()
    except: return FALLBACK_TICKERS.copy()

print("🔍 Obtendo tickers...")
tickers_b3 = obter_tickers_b3()
print(f"✅ {len(tickers_b3)} tickers.")

tickers_yahoo = [t + ".SA" for t in tickers_b3]
tickers_liquidos = []
BATCH = 50
for i in range(0, len(tickers_yahoo), BATCH):
    batch = tickers_yahoo[i:i+BATCH]
    try:
        data = yf.download(batch, period='3mo', interval='1d', group_by='ticker', progress=False, auto_adjust=True)
        for t in batch:
            if t in TICKERS_BLOQUEADOS: continue
            try:
                df = data[t].copy()
                if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.droplevel(1)
                df.columns = [c.lower() for c in df.columns]
                df.rename(columns={'close':'Close','high':'High','low':'Low','open':'Open','volume':'Volume'}, inplace=True)
                if 'Volume' in df.columns and not df.empty:
                    vol = df['Volume'].rolling(21).mean().iloc[-1]
                    px = df['Close'].iloc[-1]
                    if pd.notna(vol) and pd.notna(px) and vol >= VOLUME_MINIMO_ACAO and (vol * px) >= VOLUME_FINANCEIRO_MINIMO:
                        tickers_liquidos.append(t)
            except: continue
    except Exception as e: print(f"⚠️ Erro lote {i//BATCH}: {e}")
    time.sleep(2)

if len(tickers_liquidos) < 10:
    tickers_liquidos = FALLBACK_TICKERS.copy()
print(f"💧 {len(tickers_liquidos)} ativos líquidos.")

# ✅ DOWNLOAD 5Y + RESAMPLE COM CALENDÁRIO B3 (W-FRI, SEMANAS COMPLETAS)
print("📦 Baixando dados diários (5 anos)...")
data_d = yf.download(tickers_liquidos, period='5y', interval='1d', group_by='ticker', progress=False, auto_adjust=True)
time.sleep(2)

def resample_tf(df: pd.DataFrame, freq: str, min_days: int = 4) -> pd.DataFrame:
    if df is None or df.empty: return None
    df = df.copy()
    if not isinstance(df.index, pd.DatetimeIndex): df.index = pd.to_datetime(df.index)
    agg = {'Open':'first', 'High':'max', 'Low':'min', 'Close':'last', 'Volume':'sum'}
    df_resampled = df.resample(freq, closed='left', label='left').agg(agg)
    # Filtrar semanas com menos de min_days de negociação
    if freq.startswith('W'):
        counts = df.resample(freq, closed='left', label='left').count()['Close']
        df_resampled = df_resampled[counts >= min_days]
    return df_resampled.dropna()

data_w, data_m = {}, {}
print("⚙️ Gerando dados semanais (W-FRI) e mensais (ME)...")
for t in tickers_liquidos:
    try:
        if t in data_d and not data_d[t].empty:
            df_d = data_d[t].copy()
            if isinstance(df_d.columns, pd.MultiIndex): df_d.columns = df_d.columns.droplevel(1)
            data_w[t] = resample_tf(df_d, 'W-FRI')
            data_m[t] = resample_tf(df_d, 'ME')
    except: continue
print("✅ Dados gerados com sucesso.")

# Cache macro
cache_macro = {}
for t_ref, (bench, _) in MACRO_REFERENCE.items():
    if bench not in cache_macro:
        try:
            df_b = yf.download(bench, period='1y', interval='1wk', progress=False, auto_adjust=True)
            cache_macro[bench] = df_b['Close'].values if not df_b.empty else None
        except: cache_macro[bench] = None

# ✅ REGIME DE VOLATILIDADE + ATUALIZAÇÃO SEGURA DE PARAMS_ATIVOS
try:
    ibov = yf.download("^IBOV", period='3mo', interval='1d', progress=False)['Close']
    regime_vol = detectar_regime_volatilidade(pd.DataFrame({'Close': ibov}))
    PARAMS_ATIVOS.clear()
    PARAMS_ATIVOS.update(PARAMS_ALTA_VOL if regime_vol == 'ALTA' else PARAMS_BAIXA_VOL)
    print(f"📊 Regime: {regime_vol} (Parâmetros atualizados)")
except:
    regime_vol = 'BAIXA'

kelly_pct = fractional_kelly(WIN_RATE_ESTIMADO, PAYOFF_ESTIMADO, PARAMS_ATIVOS['kelly_frac'])
risco_maximo = CAPITAL_TOTAL * kelly_pct

# Circuit breakers
def verificar_circuit_breakers() -> Tuple[bool, str]:
    if not os.path.exists(ARQUIVO_LOG): return True, None
    try:
        with open(ARQUIVO_LOG, 'r', encoding='utf-8') as f: logs = json.load(f)
        hoje = datetime.now().date()
        trades = [l for l in logs if l['tipo'] == 'TRADE_FECHADO' and datetime.fromisoformat(l['timestamp']).date() == hoje]
        if not trades: return True, None
        pnl = sum(t['dados'].get('pnl_real', 0) for t in trades)
        dd = abs(pnl) / CAPITAL_TOTAL
        if dd >= DRAWDOWN_MAX_DIARIO: return False, f"Drawdown {dd*100:.1f}% >= {DRAWDOWN_MAX_DIARIO*100}%"
        perdas = 0
        for t in reversed(trades):
            if t['dados'].get('pnl_real', 0) < 0: perdas += 1
            else: break
        if perdas >= MAX_PERDAS_CONSECUTIVAS: return False, f"{perdas} perdas seguidas (limite: {MAX_PERDAS_CONSECUTIVAS})"
        return True, None
    except: return True, None

pode, motivo = verificar_circuit_breakers()
if not pode:
    print(f"🛑 CIRCUIT BREAKER ATIVADO: {motivo}")
    print("⏸️ Execução pausada. Retorne amanhã.")
    raise SystemExit

oportunidades_swing = []
oportunidades_position = []

def get_df(data, ticker):
    if data and ticker in data:
        df = data[ticker].copy()
        if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.droplevel(1)
        df.rename(columns={'close':'Close','high':'High','low':'Low','open':'Open','volume':'Volume'}, inplace=True)
        return df
    return None

for i, ticker in enumerate(tickers_liquidos):
    print(f"Analisando {ticker} ({i+1}/{len(tickers_liquidos)})...", end='\r')
    df_w, df_d, df_m = get_df(data_w, ticker), get_df(data_d, ticker), get_df(data_m, ticker)
    vol_fin = (df_w['Volume'] * df_w['Close']).rolling(20).mean().iloc[-1] if df_w is not None and not df_w.empty else None
    
    for mod, ref, fib, cfg in [('Swing', df_w, df_w, OTIMIZADO_SWING), ('Position', df_m, df_w, OTIMIZADO_POSITION)]:
        if ref is None or ref.empty: continue
        res = analisar_swing_trade(ticker, df_w=ref, df_d=df_d) if mod=='Swing' else analisar_position_trade(ticker, df_m=ref, df_w=fib)
        if not res: continue
        for r in res:
            e, rp = r['Entrada'], r['Risco (R$)'] / r['Entrada']
            if e < PRECO_MINIMO or rp < (0.02 if mod=='Swing' else 0.03) or rp > (0.20 if mod=='Swing' else 0.30): continue
            if EXIGIR_CONFLUENCIA and (r['Regime'] not in [1,2] or r['Eficiência'] is None or r['Eficiência'] < (0.6 if mod=='Swing' else 0.5)): continue
            mm = ref['Close'].rolling(200 if mod=='Swing' else 50).mean().iloc[-1]
            if pd.notna(mm) and ((r['Direcao']=='COMPRA' and e < mm) or (r['Direcao']=='VENDA' and e > mm)): continue
            
            # ✅ FILTRO DE VOLUME ANORMAL (Em alta volatilidade, exige volume)
            if PARAMS_ATIVOS.get('exigir_volume_anormal', False) and not r.get('Volume Anormal', True):
                log_evento('FILTRO_VOLUME', ticker, {'motivo': 'volume_normal_em_alta_vol'})
                continue
            
            align, _ = verificar_alinhamento_macro(ticker, r['Direcao'], cache_macro)
            if not align: continue
            q_vol, s_vol = avaliar_qualidade_volume(ref, r['Direcao'])
            wyck, w_conf = detectar_fase_wyckoff_adaptativo(ref, r.get('Suporte'), r.get('Resistência'))
            if (r['Direcao']=='COMPRA' and wyck=='DISTRIBUICAO') or (r['Direcao']=='VENDA' and wyck=='ACUMULACAO'): continue
            
            mult = calcular_score_confianca(r, align, q_vol, s_vol, wyck, w_conf)
            fat_liq = min(1.0, vol_fin / LIMITE_LIQUIDEZ_FINANCEIRA) if pd.notna(vol_fin) else 0.5
            lote_base = int(risco_maximo / r['Risco (R$)'])
            lote_aj = int(lote_base * mult * fat_liq)
            if lote_aj == 0: continue
            
            al_rec, met_al = calcular_alvo_recomendado(r)
            p_real = calcular_payoff_real(e, al_rec, r['Stop Loss'], PARAMS_ATIVOS['custos_pct'])
            if p_real < 2.0: continue
            
            if len(ref) >= 2:
                gap = abs(e - ref['Close'].iloc[-2]) / ref['Close'].iloc[-2]
                if gap > PARAMS_ATIVOS['gap_max_pct'] and ((r['Direcao']=='COMPRA' and e < ref['Close'].iloc[-2]) or (r['Direcao']=='VENDA' and e > ref['Close'].iloc[-2])):
                    log_evento('GAP_FILTER', ticker, {'gap': gap})
                    continue
                    
            r.update({'Alvo Recomendado': round(al_rec,2), 'Método Alvo': met_al, 'Payoff Real': p_real, 'Kelly %': round((kelly_pct * mult * fat_liq)*100,2), 'Lote': lote_aj, 'Score': int(mult*100), 'Fase Wyckoff': wyck, 'Wyckoff Conf': round(w_conf,2), 'Regime Vol': regime_vol})
            if HABILITAR_LOGGING:
                log_evento('SETUP', ticker, {'mod': mod, 'dir': r['Direcao'], 'entrada': e, 'stop': r['Stop Loss'], 'alvo': al_rec, 'score': r['Score']})
            
            if mod=='Swing': oportunidades_swing.append(r)
            else: oportunidades_position.append(r)

if len(oportunidades_position) > MAX_SETUPS_POR_DIA:
    oportunidades_position = sorted(oportunidades_position, key=lambda x: x['Score'], reverse=True)[:MAX_SETUPS_POR_DIA]

print(f"\n🎯 Swing: {len(oportunidades_swing)} | Position: {len(oportunidades_position)} setups")
print(f"   Kelly: {kelly_pct*100:.2f}% | Regime: {regime_vol}")
if oportunidades_swing: enviar_email_ou_exibir(oportunidades_swing, "Swing Trade", regime_vol)
if oportunidades_position: enviar_email_ou_exibir(oportunidades_position, "Position Trade", regime_vol)
print("\n✅ Execução concluída. Sistema 4.2 polido e pronto para produção assistida.")
